# BERT Fine-Tuning for NER on CoNLL-2003

Run cells top to bottom. **Runtime > Change runtime type > GPU (T4)** before starting.

If you re-run the install cell after already importing `transformers`, restart the runtime (**Runtime > Restart session**) before continuing, or the newer `eval_strategy` / `processing_class` arguments below may fail.

**Note on the dataset:** the original `conll2003` repo relies on a loading script, which newer versions of the `datasets` library (4.0+) no longer execute — you'd hit a `RuntimeError: Dataset scripts are no longer supported` error. This notebook uses `lhoestq/conll2003`, a script-free parquet mirror with the same train/validation/test splits and `ner_tags` schema.

In [ ]:
!pip install -q -U transformers datasets evaluate seqeval accelerate

## Restart runtime here if this is not a fresh session
`Runtime > Restart session`, then continue from the next cell.

In [ ]:
import os
import numpy as np
import torch
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

set_seed(42)

## Environment check

In [ ]:
print("=" * 70)
print("ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")
    print("Training will be significantly slower.")
    print("Go to Runtime > Change runtime type > GPU.")

print("=" * 70)

## Configuration

In [ ]:
MODEL_NAME = "bert-base-cased"

DATASET_NAME = "lhoestq/conll2003"

OUTPUT_DIR = "./ner_results"
FINAL_MODEL_DIR = "./best_bert_conll2003"

MAX_LENGTH = 128

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3

USE_FP16 = torch.cuda.is_available()

# If you hit an out-of-memory error on a free-tier Colab GPU, lower this
# BATCH_SIZE = 8

## Load CoNLL-2003 dataset

In [ ]:
print("\n" + "=" * 70)
print("LOADING CONLL-2003 DATASET")
print("=" * 70)

dataset = load_dataset(DATASET_NAME)

print(dataset)

## NER labels

In [ ]:
label_names = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
# (hardcoded: the parquet mirror stores ner_tags as plain ints, not ClassLabel, so .feature.names is unavailable)

id2label = {
    i: label for i, label in enumerate(label_names)
}

label2id = {
    label: i for i, label in enumerate(label_names)
}

print("\nNER Labels:")
print(label_names)

## Load tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

## Tokenization and label alignment

In [ ]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(
                    label[word_idx]
                )

            else:

                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print("Tokenization completed.")

## Load pretrained BERT

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

## Data collator

In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

## Evaluation metric

In [ ]:
metric = evaluate.load("seqeval")


def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = [
        [
            label_names[p]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(
            predictions,
            labels
        )
    ]

    true_labels = [
        [
            label_names[l]
            for p, l in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(
            predictions,
            labels
        )
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## Training configuration

In [ ]:
training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,

    fp16=USE_FP16,

    save_total_limit=2,

    report_to="none"
)

## Create trainer

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

print("Trainer ready.")

## Train model

In [ ]:
print("\nStarting training...\n")

trainer.train()

print("\nTraining completed successfully!")

## Final evaluation

In [ ]:
print("\nFinal Evaluation:\n")

results = trainer.evaluate()

for k, v in results.items():
    print(f"{k}: {v}")

## Save best model

In [ ]:
os.makedirs(
    FINAL_MODEL_DIR,
    exist_ok=True
)

trainer.save_model(
    FINAL_MODEL_DIR
)

tokenizer.save_pretrained(
    FINAL_MODEL_DIR
)

print("\nSaved model to:")
print(FINAL_MODEL_DIR)

## Reload saved model

In [ ]:
trained_tokenizer = AutoTokenizer.from_pretrained(
    FINAL_MODEL_DIR
)

trained_model = AutoModelForTokenClassification.from_pretrained(
    FINAL_MODEL_DIR
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

trained_model.to(device)

trained_model.eval()

print("Saved model successfully reloaded.")
print("Running on:", device)

## Test the model

In [ ]:
text = "Elon Musk founded SpaceX in California."

inputs = trained_tokenizer(
    text,
    return_tensors="pt"
)

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

with torch.no_grad():

    outputs = trained_model(
        **inputs
    )

predictions = torch.argmax(
    outputs.logits,
    dim=2
)

tokens = trained_tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print("\nPredictions:\n")

for token, pred in zip(
    tokens,
    predictions[0]
):

    print(
        token,
        "->",
        id2label[pred.item()]
    )

## Summary

In [ ]:
print("\nNER MODEL COMPLETE")